# Housing Price Prediction
## Exploratory / Development Notebook

This notebook is an exploratory sandbox that mirrors the production pipeline defined in `main.py`.
All heavy logic lives in the `src/` modules; this notebook calls those functions in the same order as the orchestrator.

Use it to
- Inspect raw and cleaned data
- Run EDA checks to validate assumptions before training
- Exercise each pipeline stage interactively
- Experiment with features and evaluate model quality

Do not use it to
- Replace the reproducible pipeline in `main.py`
- Store production logic in notebook cells
- Write production artifacts to disk from the notebook

### A) Environment setup and imports

Learning intent
- Make imports reliable regardless of how Jupyter was opened
- Ensure relative paths behave the same way for everyone
- Keep notebook code thin by calling functions from `src/` modules

In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
from __future__ import annotations

import os       # For changing the working directory to the repo root
import sys      # For manipulating sys.path so imports work wherever Jupyter started
from pathlib import Path  # Platform-independent file paths

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#import seaborn as sns
from IPython.display import display  # Displays DataFrames nicely in Jupyter

# ── Repo root alignment ──────────────────────────────────────────────────────
# Problem this solves:
#   Notebooks can start with different working directories depending on the IDE.
#   Relative paths then break unpredictably.
# Approach:
#   Search upward from the current folder until we find a folder that contains `src/`,
#   then set that as the working directory so paths match the orchestrator behaviour.
def find_repo_root(start: Path, marker_dir: str = "src", max_hops: int = 12) -> Path:
    current = start.resolve()
    for _ in range(max_hops):
        if (current / marker_dir).exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    raise RuntimeError(
        f"Could not find repo root containing '{marker_dir}/' starting from: {start}"
    )

PROJECT_ROOT = find_repo_root(Path.cwd())
os.chdir(PROJECT_ROOT)

# Ensure `import src.*` works even if Jupyter started elsewhere
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", Path.cwd())

# ── Imports from production modules ─────────────────────────────────────────
# We intentionally do NOT import from main.py.
# Each module is imported individually so we can call stages interactively.
from src.load_data  import ensure_raw_data_exists, load_raw_data
from src.clean_data import clean_dataframe
from src.schema     import REQUIRED_COLUMNS
from src.validate   import validate_dataframe
from src.train      import train_model
from src.evaluate   import evaluate_model, save_evaluation_plots
from src.infer      import run_inference
from src.utils      import save_csv, save_model

PROJECT_ROOT: C:\Users\rothl\OneDrive\IE\2_Term\MLOPS\G2_MLOPS_Project\G2_MLOPS_Project
cwd: C:\Users\rothl\OneDrive\IE\2_Term\MLOPS\G2_MLOPS_Project\G2_MLOPS_Project


### B) Sandbox configuration

Guideline
- Keep all configuration in one place at the top
- Use the same column names and paths as the orchestrator (`main.py`)
- Prefer explicit values over magic inference so errors are actionable

In [13]:
RAW_DATA_PATH  = PROJECT_ROOT / "data" / "raw"  / "Housing.csv"
PROCESSED_DIR  = PROJECT_ROOT / "data" / "processed"
MODELS_DIR     = PROJECT_ROOT / "models"
REPORTS_DIR    = PROJECT_ROOT / "reports"
INFERENCE_DIR  = PROJECT_ROOT / "data" / "inference"

TARGET_COLUMN  = "price"

# Numeric features used in EDA
NUMERIC_FEATURES = ["price", "area", "bedrooms", "bathrooms", "stories", "parking"]
BINARY_FEATURES  = ["mainroad", "guestroom", "basement", "hotwaterheating",
                    "airconditioning", "prefarea"]

### 1) Load raw data (`src.load_data`)

Educational note
- We load raw data exactly once and treat it as immutable input
- `ensure_raw_data_exists` downloads the file from Kaggle automatically when it is absent

In [14]:
ensure_raw_data_exists(RAW_DATA_PATH, fetch_if_missing=True)

df_raw = load_raw_data(RAW_DATA_PATH)
print("df_raw.shape:", df_raw.shape)
df_raw.head()

[load_data] Loading from: C:\Users\rothl\OneDrive\IE\2_Term\MLOPS\G2_MLOPS_Project\G2_MLOPS_Project\data\raw\Housing.csv
[load_data] Loaded 545 rows x 13 columns.
[load_data] Columns: ['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']
df_raw.shape: (545, 13)


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


### 2) Focused EDA checks

Goal
- Validate assumptions before investing in feature engineering and training

Guideline
- Prefer small, targeted checks over long notebooks
- If an assumption is wrong, fix the upstream module or the data contract

In [15]:
# Data types and non-null counts
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   price             545 non-null    int64
 1   area              545 non-null    int64
 2   bedrooms          545 non-null    int64
 3   bathrooms         545 non-null    int64
 4   stories           545 non-null    int64
 5   mainroad          545 non-null    str  
 6   guestroom         545 non-null    str  
 7   basement          545 non-null    str  
 8   hotwaterheating   545 non-null    str  
 9   airconditioning   545 non-null    str  
 10  parking           545 non-null    int64
 11  prefarea          545 non-null    str  
 12  furnishingstatus  545 non-null    str  
dtypes: int64(6), str(7)
memory usage: 55.5 KB


In [16]:
print(f"Dataset shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")

# Missing values
nan_count = df_raw.isna().sum()
print("\nMissing values per column:")
display(nan_count[nan_count > 0] if nan_count.sum() > 0 else "No missing values")

Dataset shape: 545 rows × 13 columns

Missing values per column:


'No missing values'

In [17]:
# Summary statistics for numeric features
df_raw.describe()

,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


In [18]:
# Target variable distribution
print(f"Target: '{TARGET_COLUMN}'")
print(f"  min  : {df_raw[TARGET_COLUMN].min():,.0f}")
print(f"  median: {df_raw[TARGET_COLUMN].median():,.0f}")
print(f"  max  : {df_raw[TARGET_COLUMN].max():,.0f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(df_raw[TARGET_COLUMN], bins=30, color="#636bb8", edgecolor="black", alpha=0.8)
axes[0].set_title("Distribution of price")
axes[0].set_xlabel("price")
axes[0].spines[["top", "right"]].set_visible(False)

# Boxplot
axes[1].boxplot(df_raw[TARGET_COLUMN], vert=False, patch_artist=True,
                boxprops=dict(facecolor="#636bb8", alpha=0.7),
                medianprops=dict(color="white", linewidth=2),
                whiskerprops=dict(color="#636bb8"),
                capprops=dict(color="#636bb8"),
                flierprops=dict(markerfacecolor="#e0d1d4", marker="o", markersize=5, alpha=0.5))
axes[1].set_title("Boxplot of price (outlier check)")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

Target: 'price'
  min  : 1,750,000
  median: 4,340,000
  max  : 13,300,000


C:\Users\rothl\AppData\Local\Temp\ipykernel_37384\2095574231.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
# Distributions of all numeric features
axes = df_raw[NUMERIC_FEATURES].hist(bins=30, figsize=(15, 10),
                                      color="#636bb8", edgecolor="black",
                                      alpha=0.85, grid=False)
for ax_row in axes:
    for ax in ax_row:
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", alpha=0.3)
plt.suptitle("Feature Distributions (raw)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

C:\Users\rothl\AppData\Local\Temp\ipykernel_37384\1409983325.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
# Correlation matrix
numeric_cols = ["area", "bedrooms", "bathrooms", "stories", "parking"]
corr = df_raw[numeric_cols + [TARGET_COLUMN]].corr()

fig, ax = plt.subplots(figsize=(10, 8))
#sns.heatmap(corr, cmap="Purples", center=0, annot=True, square=False, ax=ax)
ax.set_title("Correlation Matrix (Numeric Features + Target)", fontsize=14)
plt.tight_layout()
plt.show()

# Flag any strongly correlated feature pairs (> 0.80)
CORR_THRESHOLD = 0.80
high_corr = (corr[numeric_cols].where(
                np.triu(np.ones(corr[numeric_cols].shape), k=1).astype(bool))
             .stack().loc[lambda x: x.abs() > CORR_THRESHOLD])
print(f"Highly correlated pairs (|r| > {CORR_THRESHOLD}):")
print(high_corr if len(high_corr) else "  None found ✓")

Highly correlated pairs (|r| > 0.8):
  None found ✓


C:\Users\rothl\AppData\Local\Temp\ipykernel_37384\2658274183.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3) Clean data (`src.clean_data`)

Educational note
- Cleaning should be deterministic and stateless — it must not learn from the data
- All cleaning logic lives in `src/clean_data.py`; this cell is a thin wrapper

In [21]:
df_clean = clean_dataframe(df_raw)
print("df_clean.shape:", df_clean.shape)
df_clean.head()

[clean_data] Starting cleaning — shape: (545, 13)
[clean_data] No missing values found.
[clean_data] Binary-encoded columns: ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
[clean_data] Applied log1p transform to 'area'.
[clean_data] Cleaning complete — shape: (545, 13)
df_clean.shape: (545, 13)


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,8.912069,4,2,3,1,0,0,0,1,2,1,furnished
1,12250000,9.100637,4,4,4,1,0,0,0,1,3,0,furnished
2,12250000,9.206433,3,2,2,1,0,1,0,0,2,1,semi-furnished
3,12215000,8.922792,4,2,2,1,0,1,0,1,3,1,furnished
4,11410000,8.912069,4,1,2,1,1,1,0,1,2,0,furnished


### 4) Didactic check — what changed after cleaning?

Goal
- Make transformations visible
- Help debug unexpected column name changes or dropped columns

In [22]:
raw_cols   = list(df_raw.columns)
clean_cols = list(df_clean.columns)

removed = sorted(set(raw_cols)   - set(clean_cols))
added   = sorted(set(clean_cols) - set(raw_cols))

print("Columns removed:", removed if removed else "none")
print("Columns added  :", added   if added   else "none")
print(f"Row count: {len(df_raw)} → {len(df_clean)}")

Columns removed: none
Columns added  : none
Row count: 545 → 545


### 5) Validate data — security gate (`src.validate`)

Educational note
- Validation is a fail-fast gate: it prevents wasting time training on broken assumptions
- If validation fails, fix the upstream module or data contract — not the notebook

In [23]:
validate_dataframe(df_clean, required_columns=REQUIRED_COLUMNS)
print("Validation passed")

[validate] Running data validation checks...
[validate] All checks passed — 545 rows, 13 columns.
Validation passed


### 6) Split into training and inference holdout (`main.py` logic)

Educational note
- 50 rows are held out as an inference "smoke-test" set, mirroring `main.py`
- The remaining rows go to training / K-fold CV
- We drop the target from the inference set to simulate real-world unseen data

In [24]:
df_infer = df_clean.sample(n=50, random_state=42)
df_train = df_clean.drop(df_infer.index)

X_infer  = df_infer.drop(columns=[TARGET_COLUMN])

print(f"Training set : {df_train.shape}  (used for K-fold CV)")
print(f"Inference set: {X_infer.shape}   (target hidden — simulates production)")

Training set : (495, 13)  (used for K-fold CV)
Inference set: (50, 12)   (target hidden — simulates production)


### 7) Train (`src.train`)

Educational note
- `train_model` runs 5-fold cross-validation on the full training set
- K-fold CV provides robust performance estimates without needing a separate test set
- The returned `pipeline` encapsulates preprocessing + the model, ready for inference

In [25]:
pipeline, cv_results = train_model(df_train, TARGET_COLUMN)

print("[notebook] Training complete")
print("Pipeline steps:", [step[0] for step in pipeline.steps])

[train] Model 5: K-Fold CV on full dataset + final refit on all rows
[train] Running 5-fold CV on 495 rows...
[features] Building ColumnTransformer (unfitted)...
[features] ColumnTransformer built.
numeric: ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
categorical: ['furnishingstatus']
binary: ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
[train] Fold 1: R²=0.695  MAE=800,844  RMSE=1,188,058
[features] Building ColumnTransformer (unfitted)...
[features] ColumnTransformer built.
numeric: ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
categorical: ['furnishingstatus']
binary: ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
[train] Fold 2: R²=0.648  MAE=754,441  RMSE=1,167,481
[features] Building ColumnTransformer (unfitted)...
[features] ColumnTransformer built.
numeric: ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
categorical: ['furnishingstatus']
binary: ['mainroad', 'guestro

### 8) Evaluate (`src.evaluate`)

Educational note
- `evaluate_model` computes aggregate metrics from K-fold CV results
- All scores are on the original price scale for interpretability
- `save_evaluation_plots` writes `actual_vs_predicted.png` and `residuals.png` to the reports directory

In [26]:
metrics = evaluate_model(cv_results)
print("[notebook] K-fold CV metrics:")
for k, v in metrics.items():
    print(f"  {k:20s}: {v:.4f}")

[evaluate] Model 5 CV results(mean over 5 folds, original price scale):
R² = 0.653
Adjusted R² = 0.644
MAE = 747,580
RMSE = 1,039,102

[notebook] K-fold CV metrics:
  r2                  : 0.6533
  adjusted_r2         : 0.6440
  mae                 : 747579.7266
  rmse                : 1039102.3457


In [27]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
save_evaluation_plots(cv_results, reports_dir=REPORTS_DIR)
print(f"[notebook] Evaluation plots saved → {REPORTS_DIR}")

[evaluate] Plot saved → C:\Users\rothl\OneDrive\IE\2_Term\MLOPS\G2_MLOPS_Project\G2_MLOPS_Project\reports\actual_vs_predicted.png
[evaluate] Plot saved → C:\Users\rothl\OneDrive\IE\2_Term\MLOPS\G2_MLOPS_Project\G2_MLOPS_Project\reports\residuals.png
[notebook] Evaluation plots saved → C:\Users\rothl\OneDrive\IE\2_Term\MLOPS\G2_MLOPS_Project\G2_MLOPS_Project\reports


### 9) Inference demo (`src.infer`)

Educational note
- Inference simulates what happens in production: the pipeline sees only features, never the target
- `run_inference` applies the same preprocessing pipeline fitted during training, preventing leakage

In [28]:
predictions = run_inference(pipeline, X_infer)

print("[notebook] Predictions on 50-row holdout:")
display(predictions.head(10))

[infer] Running inference on 50 rows...
[infer] Done. Predicted price range: 2,721,399 – 11,506,818
[notebook] Predictions on 50-row holdout:


,prediction
316,4.683809e+06
77,7.290833e+06
360,3.324107e+06
90,4.603452e+06
493,3.375055e+06
209,3.503877e+06
176,5.266479e+06
249,6.420453e+06
516,2.750819e+06
426,2.721399e+06


In [29]:
# Quick sense-check: predicted price distribution vs actuals
fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(df_infer[TARGET_COLUMN], bins=20, alpha=0.6, color="#636bb8",
        label="Actual (holdout)", edgecolor="black")
ax.hist(predictions["predicted_price"] if "predicted_price" in predictions.columns
        else predictions.iloc[:, 0],
        bins=20, alpha=0.6, color="#e0d1d4",
        label="Predicted", edgecolor="black")

ax.set_xlabel("price")
ax.set_title("Actual vs Predicted — Inference Holdout (distribution check)")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

C:\Users\rothl\AppData\Local\Temp\ipykernel_37384\3971228930.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 10) Inspect production artifacts from the orchestrator

This notebook does not write artifacts to disk.

Use this cell only **after** running the full pipeline from the terminal:
```bash
python main.py
```

Goal
- Prove that the pipeline output exists on disk
- Inspect outputs without modifying them

In [30]:
from src.utils import load_model
import glob

CLEAN_DATA_PATH = PROCESSED_DIR / "clean.csv"

# Find the most recent model and prediction files
model_files = sorted(glob.glob(str(MODELS_DIR / "model_*.joblib")))
pred_files  = sorted(glob.glob(str(INFERENCE_DIR / "predictions_*.csv")))

try:
    clean_from_disk = pd.read_csv(CLEAN_DATA_PATH)
    print("clean.csv shape   :", clean_from_disk.shape)
except FileNotFoundError:
    print("clean.csv not found — run main.py first")

if model_files:
    model_from_disk = load_model(model_files[-1])
    print("Latest model      :", model_files[-1])
    print("Pipeline steps    :", [s[0] for s in model_from_disk.steps])
else:
    print("No model files found — run main.py first")

if pred_files:
    preds_from_disk = pd.read_csv(pred_files[-1])
    print("Latest predictions:", pred_files[-1])
    display(preds_from_disk.head())
else:
    print("No prediction files found — run main.py first")

clean.csv shape   : (545, 13)
Latest model      : C:\Users\rothl\OneDrive\IE\2_Term\MLOPS\G2_MLOPS_Project\G2_MLOPS_Project\models\model_20260305_125113.joblib
Pipeline steps    : ['preprocess', 'model']
Latest predictions: C:\Users\rothl\OneDrive\IE\2_Term\MLOPS\G2_MLOPS_Project\G2_MLOPS_Project\data\inference\predictions_20260305_125113.csv


,prediction
0,4.683809e+06
1,7.290833e+06
2,3.324107e+06
3,4.603452e+06
4,3.375055e+06
